<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Judges the feature-engineering pipeline by loading candidate data and inspecting implementation details and diagnostics.

**Notebook Shape:** 24 cells (15 code, 9 markdown).

**Inputs / Data Sources:**
- `df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Load source package and input path.
2. Run or inspect feature-engineering functions.
3. Review diagnostic outputs and schema changes.
4. Validate whether the pipeline is ready for training.

**Maintainability Notes:** Contains diagnostic and introspection logic; avoid using it as production orchestration without reducing it to explicit tests or scripts.


# Pipeline — Run, Judge, Trace & Test
Wired to your **real** `feature_engineering.py`. It does four things:

1. **Run** your actual `run_feature_engineering_job` and reproduce the diagnostics.
2. **Judge** the numbers automatically (fill-source mix, suspicious vs advanced-standing, stability, row counts).
3. **Trace** one student through the fallback columns.
4. **Test** the invariants that matter — including a direct check that the suspicious/advanced-standing **bug stays fixed**.

> You run with `structural_zero_as_nan=True` (that is why `model_prev_gpa` is NaN on zero-fallback rows). This notebook uses the same setting.


## Config

In [1]:
import sys
from pathlib import Path
from src.paths import FEATURES_DIR
PROJECT_ROOT = Path.cwd().parent  # if notebook is inside /notebooks
sys.path.append(str(PROJECT_ROOT))

In [2]:
from pathlib import Path
import sys, importlib
import pandas as pd, numpy as np

# --- EDIT ME: where feature_engineering.py lives (folder, not the file) ---
for cand in [".", "src", "06_feature_engineering", "/mnt/user-data/uploads"]:
    if (Path(cand) / "feature_engineering.py").exists():
        sys.path.insert(0, str(Path(cand).resolve())); break
import src.feature_engineering as fe
importlib.reload(fe)

# --- EDIT ME: the merged input to feed the job (parquet or csv) ---

CANDIDATES = [FEATURES_DIR / "selected_model_population.parquet"]
DATA_PATH = next((p for p in CANDIDATES if Path(p).exists()), None)
assert DATA_PATH, f"No input found. Tried: {CANDIDATES}"

STRUCTURAL_ZERO_AS_NAN = True   # matches how you run it
print("feature_engineering loaded from:", fe.__file__)
print("input:", DATA_PATH)


feature_engineering loaded from: D:\AI\Real projects\Academic_Advisor\src\feature_engineering.py
input: D:\AI\Real projects\Academic_Advisor\data\features\selected_model_population.parquet


In [3]:
def load_any(path):
    path = str(path)
    df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
    # drop a stray index column if the csv carried one
    for junk in ("Unnamed: 0",):
        if junk in df.columns: df = df.drop(columns=junk)
    return df

df_in = load_any(DATA_PATH)
print(f"loaded {df_in.shape[0]:,} rows x {df_in.shape[1]} cols")


loaded 761,346 rows x 28 cols


## Part 1 — Run the real job

In [4]:
result = fe.run_feature_engineering_job(df_in, structural_zero_as_nan=STRUCTURAL_ZERO_AS_NAN)
audit   = result["df_model_audit"]      # full rows + all features
primary = result["df_primary"]          # rows that feed the model (policy-clean)
excluded= result["df_excluded_over_policy"]
diag    = result["diagnostics"]
print("\nReturned:", list(result.keys()))


Original df rows: 761346
Suffix consistency check: no conflicts detected.
No semester-level conflicts detected before aggregation.
Timeline diagnostics:


,value
student_degree_timeline_count,16927.0
part_sort_key_min,20051.0
part_sort_key_max,20252.0
first_semester_concept_mismatch_count,1200.0


Semester feature merge check:


,row_count
_merge,
both,761346
left_only,0
right_only,0


Last valid GPA non-null ratio in semester frame: 0.8927297247593944
Last valid GPA non-null ratio after merge in df_model_audit: 0.8698620600883172
Previous-GPA chain diagnostics:


...

Semester stability check: all columns stable within semester groups.
Last valid GPA non-null ratio in df_primary: 0.8647197507185527
Row counts:


,row_count
original_df,761346
df_model_audit,761346
df_primary,727852
df_excluded_over_policy,33494


Final added or repaired columns:


,column
0,university_id
1,is_high_credit_course
2,over_policy_semester_credits
3,over_policy_semester_courses
4,exclude_over_policy_semester
5,is_extreme_fail_history
6,total_fail_credits_capped
7,is_interruption_semester
8,prev_semester_was_interruption
9,prior_interruption_count



Returned: ['df_model_audit', 'df_primary', 'df_excluded_over_policy', 'diagnostics']


## Part 2 — Judge the diagnostics (automatic read-out)

In [5]:
def judge(diag, audit):
    n = len(audit)
    print("="*58)
    print("FILL-SOURCE MIX")
    fs = audit["prev_gpa_fill_source"].value_counts(dropna=False)
    for k, v in fs.items():
        print(f"   {str(k):<42}{v:>9,}  ({v/n:5.1%})")
    if "last_valid_gpa_before_current_semester" not in fs.index:
        print("   note: last_valid never won as a fill source (2nd rung is dead in practice)")

    print("-"*58)
    print("ZERO-FALLBACK SPLIT")
    susp = diag.get("suspicious_zero_fallback_count")
    adv  = diag.get("advanced_standing_zero_fallback_count")
    zf   = int((audit["prev_gpa_fill_source"]=="zero_fallback").sum())
    freshmen = zf - (susp or 0) - (adv or 0)
    print(f"   zero_fallback total           {zf:>9,}")
    print(f"   - suspicious (returning, 0)   {susp:>9,}   <- review queue")
    print(f"   - advanced standing (lvl>=2)  {adv:>9,}   <- legitimate cold-start")
    print(f"   - remaining (true freshmen)   {freshmen:>9,}   <- expected")

    print("-"*58)
    print("SEMESTER STABILITY (should all be 0)")
    sc = diag.get("semester_stability_conflicts", {})
    bad = {k:v for k,v in sc.items() if v}
    print("   OK — all stable" if not bad else f"   CONFLICTS: {bad}")

    print("-"*58)
    print("ROW COUNTS")
    for k, v in diag.get("row_counts", {}).items():
        print(f"   {k:<28}{v:>9,}")
    print("="*58)

judge(diag, audit)


FILL-SOURCE MIX
   raw_prev_gpa                                633,622  (83.2%)
   zero_fallback                                95,152  (12.5%)
   last_valid_gpa_before_current_semester       31,782  ( 4.2%)
   start_agpa_points                               790  ( 0.1%)
----------------------------------------------------------
ZERO-FALLBACK SPLIT
   zero_fallback total              95,152
   - suspicious (returning, 0)         346   <- review queue
   - advanced standing (lvl>=2)      2,115   <- legitimate cold-start
   - remaining (true freshmen)      92,691   <- expected
----------------------------------------------------------
SEMESTER STABILITY (should all be 0)
   OK — all stable
----------------------------------------------------------
ROW COUNTS
   original_df                   761,346
   df_model_audit                761,346
   df_primary                    727,852
   df_excluded_over_policy        33,494


## Part 3 — Trace one student

In [6]:
TRACE_COLS = [
    "part_id", "course_id",
    "prev_gpa_points", "last_valid_gpa_before_current_semester", "start_agpa_points",
    "prev_gpa_points_clean", "prev_gpa_fill_source",
    "model_prev_gpa",
    "is_first_active_semester", "start_level_ord",
    "prev_gpa_invalid_zero_case",
]

# default: a student that actually exists in the loaded data
CASE_ID = str(audit["student_id"].iloc[0])

def trace(df, case_id, cols):
    cols = [c for c in cols if c in df.columns]
    sub = df[df["student_id"].astype(str) == str(case_id)].copy()
    if "part_id" in sub.columns:
        sub["__k"] = pd.to_numeric(sub["part_id"], errors="coerce")
        sub = sub.sort_values("__k").drop(columns="__k")
    print(f"student {case_id}: {len(sub)} rows")
    return sub[cols].reset_index(drop=True)

trace(audit, CASE_ID, TRACE_COLS)


student 10000.111: 60 rows


,part_id,course_id,prev_gpa_points,last_valid_gpa_before_current_semester,start_agpa_points,prev_gpa_points_clean,prev_gpa_fill_source,model_prev_gpa,is_first_active_semester,start_level_ord,prev_gpa_invalid_zero_case
0,20151,432.111,<NA>,<NA>,0.0,0.00,zero_fallback,NaN,1,1,0
1,20151,501.111,<NA>,<NA>,0.0,0.00,zero_fallback,NaN,1,1,0
2,20151,513.111,<NA>,<NA>,0.0,0.00,zero_fallback,NaN,1,1,0
3,20151,967.111,<NA>,<NA>,0.0,0.00,zero_fallback,NaN,1,1,0
4,20151,955.111,<NA>,<NA>,0.0,0.00,zero_fallback,NaN,1,1,0
5,20151,956.111,<NA>,<NA>,0.0,0.00,zero_fallback,NaN,1,1,0
6,20152,502.111,2.41,2.41,2.41,2.41,raw_prev_gpa,2.41,0,1,0
7,20152,1016.111,2.41,2.41,2.41,2.41,raw_prev_gpa,2.41,0,1,0
8,20152,520.111,2.41,2.41,2.41,2.41,raw_prev_gpa,2.41,0,1,0
9,20152,514.111,2.41,2.41,2.41,2.41,raw_prev_gpa,2.41,0,1,0


### Test a single hypothetical case (mirrors `repair_previous_gpa_chain` row logic)
Pass `None` for a missing (NA) value. Lets you probe the fallback without touching the data.

In [7]:
def simulate_fallback(raw, last_valid, start_agpa, is_first_active, start_level_ord,
                      structural_zero_as_nan=STRUCTURAL_ZERO_AS_NAN):
    clean, source = None, None
    for name, val in [("raw_prev_gpa", raw),
                      ("last_valid_gpa_before_current_semester", last_valid),
                      ("start_agpa_points", start_agpa)]:
        if clean is None and val is not None and val > 0:
            clean, source = val, name
    if clean is None:
        clean, source = 0.0, "zero_fallback"
    invalid_zero = int(raw == 0 and is_first_active != 1)
    model = (None if (structural_zero_as_nan and source == "zero_fallback") else clean)
    zf = source == "zero_fallback"
    return {
        "prev_gpa_points_clean": clean,
        "prev_gpa_fill_source": source,
        "prev_gpa_points_missing": int(raw is None),
        "prev_gpa_invalid_zero_case": invalid_zero,
        "model_prev_gpa": model,
        "suspicious": bool(zf and is_first_active == 0),
        "advanced_standing": bool(zf and is_first_active == 1 and start_level_ord >= 2),
    }

# example: an advanced-standing transfer entrant (no prior GPA, entered at level 2)
simulate_fallback(raw=None, last_valid=None, start_agpa=0.0,
                  is_first_active=1, start_level_ord=2)


{'prev_gpa_points_clean': 0.0,
 'prev_gpa_fill_source': 'zero_fallback',
 'prev_gpa_points_missing': 1,
 'prev_gpa_invalid_zero_case': 0,
 'model_prev_gpa': None,
 'suspicious': False,
 'advanced_standing': True}

## Part 4 — Tests (invariants + the bug-fix guard)

In [8]:
PASS, FAIL, SKIP = [], [], []
def check(name, cond, detail=""):
    if cond is None: SKIP.append(name); print(f"  SKIP  {name}  {detail}")
    elif cond:       PASS.append(name); print(f"  PASS  {name}")
    else:            FAIL.append(name); print(f"  FAIL  {name}  {detail}")


In [9]:
# T1 — leakage guard actually works on a built feature matrix
keep = [c for c in audit.columns if c not in set(fe.LEAKAGE_COLUMNS)]
X = audit[keep]
try:
    fe.assert_no_leakage_columns(X); leak_ok = True
except ValueError: leak_ok = False
check("assert_no_leakage_columns passes on clean X", leak_ok)

# and it must RAISE when a leakage column is present
try:
    fe.assert_no_leakage_columns(audit[keep + ["part_id"]]); raised = False
except ValueError: raised = True
check("leakage guard raises when part_id is injected", raised)


  PASS  assert_no_leakage_columns passes on clean X
  PASS  leakage guard raises when part_id is injected


In [10]:
# T2 — fill source only ever uses the known rungs
ALLOWED = {"raw_prev_gpa","last_valid_gpa_before_current_semester","start_agpa_points","zero_fallback"}
seen = set(audit["prev_gpa_fill_source"].dropna().unique())
check("fill_source within allowed rungs", seen <= ALLOWED, f"unexpected: {seen-ALLOWED}")

# no nulls / NaN in the repaired columns (the job asserts this too)
check("prev_gpa_fill_source has no nulls", int(audit['prev_gpa_fill_source'].isna().sum()) == 0)
check("prev_gpa_points_clean has no NaN",  int(audit['prev_gpa_points_clean'].isna().sum()) == 0)


  PASS  fill_source within allowed rungs
  PASS  prev_gpa_fill_source has no nulls
  PASS  prev_gpa_points_clean has no NaN


In [11]:
# T3 — THE BUG-FIX GUARD: suspicious and advanced-standing must never overlap
zf  = audit["prev_gpa_fill_source"].eq("zero_fallback")
lvl = pd.to_numeric(audit.get("start_level_ord", 0), errors="coerce").fillna(0)
susp_mask = zf & audit["is_first_active_semester"].eq(0)
adv_mask  = zf & audit["is_first_active_semester"].eq(1) & lvl.ge(2)

overlap = int((susp_mask & adv_mask).sum())
check("suspicious and advanced-standing are mutually exclusive", overlap == 0,
      f"{overlap} rows flagged as BOTH")

# every suspicious row is a returning student; every advanced row is first-active & level>=2
check("all suspicious rows are returning (is_first_active==0)",
      bool((audit.loc[susp_mask, "is_first_active_semester"] == 0).all()) if susp_mask.any() else None)
check("all advanced rows are first-active & level>=2",
      bool((audit.loc[adv_mask, "is_first_active_semester"].eq(1)).all()) if adv_mask.any() else None)

# counts match the diagnostics the job reported
check("suspicious count matches diagnostics",
      int(susp_mask.sum()) == diag.get("suspicious_zero_fallback_count"))
check("advanced count matches diagnostics",
      int(adv_mask.sum()) == diag.get("advanced_standing_zero_fallback_count"))


  PASS  suspicious and advanced-standing are mutually exclusive
  PASS  all suspicious rows are returning (is_first_active==0)
  PASS  all advanced rows are first-active & level>=2
  PASS  suspicious count matches diagnostics
  PASS  advanced count matches diagnostics


In [12]:
# T4 — model_prev_gpa is NaN exactly on zero-fallback rows (structural_zero_as_nan=True)
if STRUCTURAL_ZERO_AS_NAN and "model_prev_gpa" in audit.columns:
    nan_mask = audit["model_prev_gpa"].isna()
    check("model_prev_gpa is NaN exactly where fill_source==zero_fallback",
          bool((nan_mask == zf).all()), "mismatch between NaN mask and zero_fallback mask")
else:
    check("model_prev_gpa is NaN exactly where fill_source==zero_fallback", None,
          "needs structural_zero_as_nan=True")


  PASS  model_prev_gpa is NaN exactly where fill_source==zero_fallback


In [13]:
# T5 — semester stability: every checked column constant within a semester group
sc = diag.get("semester_stability_conflicts", {})
check("no semester-stability conflicts", sum(sc.values()) == 0 if sc else None, f"{sc}")

# row-count identity holds
rc = diag.get("row_counts", {})
if rc:
    check("primary + excluded == audit",
          rc["df_primary"] + rc["df_excluded_over_policy"] == rc["df_model_audit"])
else:
    check("primary + excluded == audit", None)


  PASS  no semester-stability conflicts
  PASS  primary + excluded == audit


In [14]:
print("\n" + "="*40)
print(f"PASS : {len(PASS)}")
print(f"FAIL : {len(FAIL)}   {FAIL}")
print(f"SKIP : {len(SKIP)}   {SKIP}")
print("="*40)
assert not FAIL, f"{len(FAIL)} test(s) failed: {FAIL}"
print("All active tests passed.")



PASS : 13
FAIL : 0   []
SKIP : 0   []
All active tests passed.


## Part 5 — Function map (the real module, in run order)

In [15]:
import inspect
RUN_ORDER = [
    "ensure_university_id", "normalize_timeline_keys", "add_policy_and_fail_flags",
    "build_semester_history", "merge_semester_history", "repair_previous_gpa_chain",
    "add_remaining_rowwise_features", "report_suspicious_zero_fallback_rows",
    "check_semester_stability",
]
print(f"{'#':>2}  {'function':<38}{'lines':>6}")
print("-"*48)
for i, name in enumerate(RUN_ORDER, 1):
    fn = getattr(fe, name, None)
    ln = len(inspect.getsourcelines(fn)[0]) if fn else 0
    print(f"{i:>2}  {name:<38}{ln:>6}")


 #  function                               lines
------------------------------------------------
 1  ensure_university_id                      54
 2  normalize_timeline_keys                   12
 3  add_policy_and_fail_flags                 31
 4  build_semester_history                   129
 5  merge_semester_history                    80
 6  repair_previous_gpa_chain                 75
 7  add_remaining_rowwise_features            10
 8  report_suspicious_zero_fallback_rows      61
 9  check_semester_stability                  58


---
**Pairs with** `pipeline_function_explorer.html` — the interactive version: feed inputs to the
`prev_gpa` fallback and watch the rung fire and the suspicious / advanced-standing flags flip live.